In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive




```
Hybrid Fusion – Liveness Detection
=====================================
A 3-stage cascade that combines all three fusion paradigms in one pipeline:

  ┌─────────────────────────────────────────────────────────────┐
  │  STAGE 1  │  Feature-Level                                  │
  │           │  Two meta-classifiers (RF + LogReg) each learn  │
  │           │  a decision boundary in probability space       │
  ├─────────────────────────────────────────────────────────────┤
  │  STAGE 2  │  Score-Level                                    │
  │           │  Weighted blend of Stage-1 outputs + raw        │
  │           │  unimodal probabilities → refined spoof score   │
  ├─────────────────────────────────────────────────────────────┤
  │  STAGE 3  │  Decision-Level                                 │
  │           │  Majority / unanimous vote across Stage-1 and   │
  │           │  Stage-2 predictions → final binary decision    │
  └─────────────────────────────────────────────────────────────┘

Bimodal pairs:
  • Face + Voice
  • Face + Keystroke
  • Voice + Keystroke

Trimodal:
  • Face + Voice + Keystroke

Evaluation: 5-Fold Stratified Cross-Validation (n=95)

Input  : fusion_val_predictions.csv
Outputs: hybrid_fusion_results.csv
         hybrid_fusion_summary.csv
```



In [2]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

from sklearn.ensemble        import RandomForestClassifier
from sklearn.linear_model    import LogisticRegression
from sklearn.preprocessing   import StandardScaler
from sklearn.pipeline        import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics         import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix, classification_report
)

In [3]:
# ──────────────────────────────────────────────────────────────
# 1.  LOAD DATA
# ──────────────────────────────────────────────────────────────
df  = pd.read_csv("/content/drive/MyDrive/Final_Dataset/Sam_final_dataset/Predictions/fusion_val_predictions.csv")
y   = df["true_label"].values
SKF = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("=" * 65)
print("HYBRID FUSION – LIVENESS DETECTION")
print("3-Stage Pipeline: Feature → Score → Decision")
print("=" * 65)
print(f"Total samples    : {len(df)}")
print(f"Genuine (0)      : {(y == 0).sum()}")
print(f"Spoof   (1)      : {(y == 1).sum()}")
print(f"Attack types     : {df['attack_type'].value_counts().to_dict()}")
print(f"Evaluation       : 5-Fold Stratified Cross-Validation\n")

print("Pipeline Architecture")
print("─" * 65)
print("  STAGE 1 │ Feature-Level  : RF meta-classifier + LogReg meta-classifier")
print("  STAGE 2 │ Score-Level    : Weighted blend (Stage1-RF × 0.5 +")
print("          │                  Stage1-LR × 0.3 + Raw avg × 0.2)")
print("  STAGE 3 │ Decision-Level : Majority vote across Stage1-RF,")
print("          │                  Stage1-LR, Stage2 predictions")
print("─" * 65)

HYBRID FUSION – LIVENESS DETECTION
3-Stage Pipeline: Feature → Score → Decision
Total samples    : 95
Genuine (0)      : 24
Spoof   (1)      : 71
Attack types     : {'tts': 25, 'genuine': 24, 'logical': 17, 'replay': 15, 'synthetic': 14}
Evaluation       : 5-Fold Stratified Cross-Validation

Pipeline Architecture
─────────────────────────────────────────────────────────────────
  STAGE 1 │ Feature-Level  : RF meta-classifier + LogReg meta-classifier
  STAGE 2 │ Score-Level    : Weighted blend (Stage1-RF × 0.5 +
          │                  Stage1-LR × 0.3 + Raw avg × 0.2)
  STAGE 3 │ Decision-Level : Majority vote across Stage1-RF,
          │                  Stage1-LR, Stage2 predictions
─────────────────────────────────────────────────────────────────


In [4]:
# ──────────────────────────────────────────────────────────────
# 2.  META-CLASSIFIER PIPELINES  (reused across all combos)
# ──────────────────────────────────────────────────────────────
def make_rf():
    return Pipeline([
        ("scaler", StandardScaler()),
        ("clf",    RandomForestClassifier(
            n_estimators=100,
            class_weight="balanced",
            random_state=42,
        )),
    ])

def make_lr():
    return Pipeline([
        ("scaler", StandardScaler()),
        ("clf",    LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42,
        )),
    ])

In [5]:
# ──────────────────────────────────────────────────────────────
# 3.  FUSION COMBINATIONS
# ──────────────────────────────────────────────────────────────
COMBOS = {
    "Face + Voice": {
        "cols":  ["face_prob_spoof", "voice_prob_spoof"],
        "level": "Bimodal",
        # Stage 2 weights: [RF_weight, LR_weight, raw_avg_weight]
        "w":     (0.50, 0.30, 0.20),
        # Stage 3 vote threshold (out of 3 streams)
        "vote_thresh": 2,      # majority (≥ 2/3)
    },
    "Face + Keystroke": {
        "cols":  ["face_prob_spoof", "keystroke_prob_spoof"],
        "level": "Bimodal",
        "w":     (0.50, 0.30, 0.20),
        "vote_thresh": 2,
    },
    "Voice + Keystroke": {
        "cols":  ["voice_prob_spoof", "keystroke_prob_spoof"],
        "level": "Bimodal",
        "w":     (0.50, 0.30, 0.20),
        "vote_thresh": 2,
    },
    "Trimodal (F+V+K)": {
        "cols":  ["face_prob_spoof", "voice_prob_spoof", "keystroke_prob_spoof"],
        "level": "Trimodal",
        "w":     (0.45, 0.35, 0.20),
        "vote_thresh": 3,      # unanimous (≥ 3/3) — strictest, lowest FAR
    },
}


In [6]:
# ──────────────────────────────────────────────────────────────
# 4.  RUN HYBRID PIPELINE FOR EACH COMBO
# ──────────────────────────────────────────────────────────────
summary_rows  = []
result_frames = []

def metrics(y_true, y_pred, y_prob):
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec  = recall_score(y_true, y_pred,    zero_division=0)
    f1   = f1_score(y_true, y_pred,        zero_division=0)
    auc  = roc_auc_score(y_true, y_prob)
    cm   = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    far  = fp / (fp + tn) if (fp + tn) > 0 else 0
    frr  = fn / (fn + tp) if (fn + tp) > 0 else 0
    return dict(acc=acc, prec=prec, rec=rec, f1=f1, auc=auc,
                tn=int(tn), fp=int(fp), fn=int(fn), tp=int(tp),
                far=far, frr=frr)

for combo_name, cfg in COMBOS.items():

    X       = df[cfg["cols"]].values
    w_rf, w_lr, w_raw = cfg["w"]
    thresh  = cfg["vote_thresh"]

    print(f"\n{'═' * 65}")
    print(f"  {cfg['level'].upper()}  ▸  {combo_name}")
    print(f"  Features  : {cfg['cols']}")
    print(f"  S2 weights: RF={w_rf}  LR={w_lr}  Raw={w_raw}")
    print(f"  S3 threshold: ≥{thresh}/3 votes")
    print(f"{'═' * 65}")

    # ── STAGE 1: Feature-Level ──────────────────────────────
    s1_rf_prob = cross_val_predict(
        make_rf(), X, y, cv=SKF, method="predict_proba")[:, 1]
    s1_lr_prob = cross_val_predict(
        make_lr(), X, y, cv=SKF, method="predict_proba")[:, 1]

    s1_rf_pred = (s1_rf_prob >= 0.5).astype(int)
    s1_lr_pred = (s1_lr_prob >= 0.5).astype(int)

    m_rf = metrics(y, s1_rf_pred, s1_rf_prob)
    m_lr = metrics(y, s1_lr_pred, s1_lr_prob)

    print(f"\n  STAGE 1 │ Feature-Level")
    print(f"    RF  meta-clf  →  Acc={m_rf['acc']*100:.2f}%  "
          f"F1={m_rf['f1']*100:.2f}%  AUC={m_rf['auc']:.4f}  "
          f"FAR={m_rf['far']*100:.2f}%  FRR={m_rf['frr']*100:.2f}%")
    print(f"    LR  meta-clf  →  Acc={m_lr['acc']*100:.2f}%  "
          f"F1={m_lr['f1']*100:.2f}%  AUC={m_lr['auc']:.4f}  "
          f"FAR={m_lr['far']*100:.2f}%  FRR={m_lr['frr']*100:.2f}%")

    # ── STAGE 2: Score-Level blend ──────────────────────────
    raw_avg   = X.mean(axis=1)
    s2_prob   = w_rf * s1_rf_prob + w_lr * s1_lr_prob + w_raw * raw_avg
    s2_pred   = (s2_prob >= 0.5).astype(int)
    m_s2      = metrics(y, s2_pred, s2_prob)

    print(f"\n  STAGE 2 │ Score-Level Blend")
    print(f"    Fused score   →  Acc={m_s2['acc']*100:.2f}%  "
          f"F1={m_s2['f1']*100:.2f}%  AUC={m_s2['auc']:.4f}  "
          f"FAR={m_s2['far']*100:.2f}%  FRR={m_s2['frr']*100:.2f}%")

    # ── STAGE 3: Decision-Level vote ────────────────────────
    votes     = s1_rf_pred + s1_lr_pred + s2_pred
    s3_pred   = (votes >= thresh).astype(int)
    s3_prob   = s2_prob                        # carry Stage-2 prob as soft score
    m_s3      = metrics(y, s3_pred, s3_prob)

    print(f"\n  STAGE 3 │ Decision-Level Vote  (≥{thresh}/3)")
    print(f"    Final output  →  Acc={m_s3['acc']*100:.2f}%  "
          f"F1={m_s3['f1']*100:.2f}%  AUC={m_s3['auc']:.4f}  "
          f"FAR={m_s3['far']*100:.2f}%  FRR={m_s3['frr']*100:.2f}%")
    print(f"    Confusion     →  TN={m_s3['tn']}  FP={m_s3['fp']}  "
          f"FN={m_s3['fn']}  TP={m_s3['tp']}")
    print(f"\n  Classification Report (Final Stage 3):")
    print(classification_report(y, s3_pred,
                                target_names=["Genuine", "Spoof"], digits=4))

    # Save per-stage summary rows
    for stage_name, m, pred, prob in [
        (f"Stage 1 – RF  ({combo_name})",  m_rf, s1_rf_pred, s1_rf_prob),
        (f"Stage 1 – LR  ({combo_name})",  m_lr, s1_lr_pred, s1_lr_prob),
        (f"Stage 2 – Score ({combo_name})",m_s2, s2_pred,    s2_prob),
        (f"Stage 3 – Final ({combo_name})",m_s3, s3_pred,    s3_prob),
    ]:
        summary_rows.append({
            "Level":          cfg["level"],
            "Fusion Combo":   combo_name,
            "Stage":          stage_name,
            "Accuracy (%)":   round(m["acc"]  * 100, 2),
            "Precision (%)":  round(m["prec"] * 100, 2),
            "Recall (%)":     round(m["rec"]  * 100, 2),
            "F1-Score (%)":   round(m["f1"]   * 100, 2),
            "AUC-ROC":        round(m["auc"], 4),
            "FAR (%)":        round(m["far"]  * 100, 2),
            "FRR (%)":        round(m["frr"]  * 100, 2),
            "TP": m["tp"], "TN": m["tn"], "FP": m["fp"], "FN": m["fn"],
        })

    # Per-sample result frame (final stage only)
    tmp = df[["session_token", "attack_type", "true_label"]].copy()
    tmp["hybrid_stage1_rf_prob"]  = s1_rf_prob.round(6)
    tmp["hybrid_stage1_rf_pred"]  = s1_rf_pred
    tmp["hybrid_stage1_lr_prob"]  = s1_lr_prob.round(6)
    tmp["hybrid_stage1_lr_pred"]  = s1_lr_pred
    tmp["hybrid_stage2_prob"]     = s2_prob.round(6)
    tmp["hybrid_stage2_pred"]     = s2_pred
    tmp["hybrid_stage3_pred"]     = s3_pred
    tmp["fusion_combo"]           = combo_name
    result_frames.append(tmp)


═════════════════════════════════════════════════════════════════
  BIMODAL  ▸  Face + Voice
  Features  : ['face_prob_spoof', 'voice_prob_spoof']
  S2 weights: RF=0.5  LR=0.3  Raw=0.2
  S3 threshold: ≥2/3 votes
═════════════════════════════════════════════════════════════════

  STAGE 1 │ Feature-Level
    RF  meta-clf  →  Acc=94.74%  F1=96.50%  AUC=0.9844  FAR=12.50%  FRR=2.82%
    LR  meta-clf  →  Acc=92.63%  F1=94.96%  AUC=0.9812  FAR=8.33%  FRR=7.04%

  STAGE 2 │ Score-Level Blend
    Fused score   →  Acc=95.79%  F1=97.18%  AUC=0.9853  FAR=8.33%  FRR=2.82%

  STAGE 3 │ Decision-Level Vote  (≥2/3)
    Final output  →  Acc=95.79%  F1=97.18%  AUC=0.9853  FAR=8.33%  FRR=2.82%
    Confusion     →  TN=22  FP=2  FN=2  TP=69

  Classification Report (Final Stage 3):
              precision    recall  f1-score   support

     Genuine     0.9167    0.9167    0.9167        24
       Spoof     0.9718    0.9718    0.9718        71

    accuracy                         0.9579        95
   macr

In [7]:
# ──────────────────────────────────────────────────────────────
# 5.  FULL COMPARISON  (all methods side by side)
# ──────────────────────────────────────────────────────────────
print("\n" + "=" * 65)
print("COMPLETE FUSION METHOD COMPARISON")
print("(Score-Level  vs  Feature-Level  vs  Hybrid)")
print("=" * 65)

# Score-level from fusion_val_predictions.csv
score_refs = [
    ("Score  │ Face+Voice",       "fv_pred",  "fv_prob_spoof"),
    ("Score  │ Face+Keystroke",   "fk_pred",  "fk_prob_spoof"),
    ("Score  │ Voice+Keystroke",  "vk_pred",  "vk_prob_spoof"),
    ("Score  │ Trimodal",         "fvk_pred", "fvk_prob_spoof"),
]

# Feature-level best results (pre-computed from feature_level_fusion.py)
feature_refs = [
    ("Feature│ Face+Voice       (RF)", 94.74, 96.50, 0.9844, 12.50, 2.82),
    ("Feature│ Face+Keystroke   (RF)", 97.89, 98.61, 0.9971,  8.33, 0.00),
    ("Feature│ Voice+Keystroke  (RF)", 93.68, 95.83, 0.9487, 16.67, 2.82),
    ("Feature│ Trimodal        (LR)",  96.84, 97.87, 0.9947,  4.17, 2.82),
]

# Hybrid final results (from summary_rows, stage 3 only)
summary_df = pd.DataFrame(summary_rows)
hybrid_finals = summary_df[summary_df["Stage"].str.startswith("Stage 3")]

print(f"\n{'Method':<42} {'Acc':>7}  {'F1':>7}  {'AUC':>7}  {'FAR':>6}  {'FRR':>6}")
print("─" * 77)

print("  SCORE-LEVEL:")
for label, pred_col, prob_col in score_refs:
    m = metrics(y, df[pred_col].values, df[prob_col].values)
    print(f"  {label:<40} {m['acc']*100:>6.2f}%  {m['f1']*100:>6.2f}%  "
          f"{m['auc']:>7.4f}  {m['far']*100:>5.2f}%  {m['frr']*100:>5.2f}%")

print("\n  FEATURE-LEVEL (best classifier per combo):")
for label, acc, f1, auc, far, frr in feature_refs:
    print(f"  {label:<40} {acc:>6.2f}%  {f1:>6.2f}%  "
          f"{auc:>7.4f}  {far:>5.2f}%  {frr:>5.2f}%")

print("\n  HYBRID (Stage 3 final — all 3 fusion types combined):")
for _, row in hybrid_finals.iterrows():
    label = f"Hybrid │ {row['Fusion Combo']}"
    print(f"  {label:<40} {row['Accuracy (%)']:>6.2f}%  {row['F1-Score (%)']:>6.2f}%  "
          f"{row['AUC-ROC']:>7.4f}  {row['FAR (%)']:>5.2f}%  {row['FRR (%)']:>5.2f}%")


COMPLETE FUSION METHOD COMPARISON
(Score-Level  vs  Feature-Level  vs  Hybrid)

Method                                         Acc       F1      AUC     FAR     FRR
─────────────────────────────────────────────────────────────────────────────
  SCORE-LEVEL:
  Score  │ Face+Voice                       92.63%   94.96%   0.9765   8.33%   7.04%
  Score  │ Face+Keystroke                   95.79%   97.14%   0.9894   4.17%   4.23%
  Score  │ Voice+Keystroke                  90.53%   93.53%   0.9783  12.50%   8.45%
  Score  │ Trimodal                         96.84%   97.87%   0.9959   4.17%   2.82%

  FEATURE-LEVEL (best classifier per combo):
  Feature│ Face+Voice       (RF)            94.74%   96.50%   0.9844  12.50%   2.82%
  Feature│ Face+Keystroke   (RF)            97.89%   98.61%   0.9971   8.33%   0.00%
  Feature│ Voice+Keystroke  (RF)            93.68%   95.83%   0.9487  16.67%   2.82%
  Feature│ Trimodal        (LR)             96.84%   97.87%   0.9947   4.17%   2.82%

  HYBRID (Stag

In [8]:
import os

# ──────────────────────────────────────────────────────────────
# 6.  SAVE OUTPUTS
# ──────────────────────────────────────────────────────────────
output_dir = "/content/drive/MyDrive/Final_Dataset/Sam_final_dataset/Predictions"
os.makedirs(output_dir, exist_ok=True)

results_df = pd.concat(result_frames, ignore_index=True)
results_df.to_csv(os.path.join(output_dir, "hybrid_fusion_results.csv"), index=False)

summary_df = summary_df.sort_values(
    ["Level", "Fusion Combo", "Stage"],
    ascending=True
).reset_index(drop=True)
summary_df.to_csv(os.path.join(output_dir, "hybrid_fusion_summary.csv"), index=False)

print("\n")
print(f"✓  Saved: {output_dir}/hybrid_fusion_results.csv")
print(f"✓  Saved: {output_dir}/hybrid_fusion_summary.csv")
print("=" * 65)



✓  Saved: /content/drive/MyDrive/Final_Dataset/Sam_final_dataset/Predictions/hybrid_fusion_results.csv
✓  Saved: /content/drive/MyDrive/Final_Dataset/Sam_final_dataset/Predictions/hybrid_fusion_summary.csv
